# Seeing ViT Registers

An independent, inference-only reproduction of [Vision Transformers Need Registers](https://arxiv.org/abs/2309.16588).
This notebook calls the implementation in `src/`; all scientific settings are in `config.yaml`.
It recomputes both Large and Base model pairs on the fixed 20-image COCO subset and regenerates every figure.

**Colab:** select a GPU runtime, then run all cells. The setup cell clones the public repository and installs
its exact requirements. **Local:** install `requirements.txt` first and open this notebook from the repository root.
The original A100 run found 21 high-norm Large no-register patches (0.0767%) versus zero with registers;
the across-layer relative-tail plot did not reproduce a clean middle-layer onset.

In [ ]:
# Environment setup only; the experiment itself is implemented in src/.
from pathlib import Path
import os
import subprocess
import sys
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    checkout = Path('/content/seeing-vit-registers')
    if not (checkout / 'src/pipeline.py').exists():
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/turhancan97/seeing-vit-registers.git', str(checkout)], check=True)
    os.chdir(checkout)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
assert Path('src/pipeline.py').exists(), 'Start the local notebook from the repository root.'
sys.path.insert(0, str(Path.cwd()))

## Reproduce all experiments

The following call downloads missing images, verifies SHA-256 hashes, loads pinned checkpoints with eager
attention, performs fresh float32 inference, writes all tables, and saves 300 dpi PNGs and companion PDFs.
CLS and all register tokens are removed before any spatial reshape; exactly 37×37 patches are asserted.
No saved feature arrays are reused by default.

In [ ]:
from src.pipeline import run_all
from src.notebook import show_results, show_figures

results = run_all()

## Results and feature spaces

Primary norms and PCA use the normalized final model output, `last_hidden_state`. The raw final-block
output, `hidden_states[-1]`, is recorded separately. These norms are not interchangeable; the pipeline
checks that the model's final LayerNorm connects the two.

The frozen cutoff 68.01323582036801 was derived from the Large no-register gap (55.7261–83.0095).
Its estimation rule, fixed sensitivity cutoffs (60/80/150), layer k=3, and seed 42 are in `config.yaml`.
These choices are exploratory rather than preregistered. Base has no clear separate mode and uses
the Large cutoff for the common-scale comparison.

In [ ]:
show_results(results)

## A — Final patch-token norm distributions

All images are pooled. Histogram y-axes are logarithmic. The LayerNorm diagnostics show why the selected
feature space matters; no assumption is made that every raw residual-stream tail is an artifact.

In [ ]:
show_figures(results, ['token_norm_histogram_large', 'token_norm_histogram_base',
                       'layernorm_diagnostic_large', 'layernorm_diagnostic_base'])

## B — Across transformer layers

Each model/layer uses its own mean + 3×population std across all images (`ddof=0`); layer 0 is the embedding.
This relative-tail metric does not reproduce a clean middle-layer onset or consistently lower register-model
rates. It is not the same population as the final normalized-output high-norm group.

In [ ]:
show_figures(results, ['high_norm_across_layers_large', 'high_norm_across_layers_base'])

## C — Spatial comparisons

Six examples were selected from the input contact sheet before model outcomes. Norm maps share a linear
scale across models and images; CLS attention uses shared log probability scales. Independent PCA fits
map the first three components to RGB; colors are not aligned semantic labels across models.

In [ ]:
show_figures(results, ['main_comparison', 'visual_comparison_large', 'visual_comparison_base',
                       'high_norm_locations_large', 'high_norm_locations_base'])

## FeatLens and the location proxy

The attributed original FeatLens kernels compute cosine similarity to the center patch and attention rollout.
Rollout retains all CLS/register transitions until the final spatial slice, averages heads, adds residual identity,
row-normalizes, and discards no edges. Rollout display scales are logarithmic and shared.

The COCO object-mask complement is an imperfect background proxy. On the original run, 19/21 Large
outliers (90.48%) were outside annotated objects; 68.42% of all patch centers were outside. Inspect the
proxy mask figure rather than interpreting this complement as semantic ground truth.

In [ ]:
show_figures(results, ['featlens_comparison_large', 'featlens_comparison_base',
                       'background_proxy_masks', 'dataset_contact_sheet'])

## Limitations and artifacts

Only 20 images, two model sizes, one resolution, no retraining, no seed/checkpoint sweep. Patches within
images are correlated. Thresholds and feature scaling affect the conclusions; the roughly 2× norm contrast
is smaller than the paper's illustrative giant-model effect. Public checkpoints do not isolate every training
difference, and visualizations are not causal evidence. Cross-platform bitwise agreement is not guaranteed.

All numerical CSV/JSON tables and PNG/PDF figures are in `results/`. Large NPZ arrays are generated locally
and ignored by Git. Hardware/package versions, exact settings and source hashes, checkpoint revisions,
and artifact validation are recorded under `results/data/`. Source-photo licenses are in `images/SOURCES.md`.
See the README and CITATION.cff for references and attribution.